In [3]:
import subprocess, time

# 1. Instalar dependência zstd (necessária para o instalador do Ollama)
!apt-get install -y zstd -q

# 2. Instalar o servidor Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Iniciar o servidor em background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Aguardar o servidor subir (importante!)
time.sleep(5)
print("🟢 Servidor Ollama iniciado!")

# 4. Instalar o SDK Python
!pip install ollama -q

# 5. Baixar o modelo base (só na primeira vez, ~800MB)
!ollama pull llama3.2:1b

# 6. Verificar
import ollama
print("✅ Tudo pronto! Servidor rodando e modelo baixado.")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1min 3s (9,564 B/s)
Selecting previously unselected package zstd.
(Reading database ... 118194 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama u

In [4]:
import subprocess
import time
import os

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(15)

In [6]:
!ollama pull phi3

In [7]:
dataset = [
    {"pergunta":"Estou com febre","resposta":"Você também apresenta tosse, falta de ar ou dor no corpo?"},
    {"pergunta":"Estou com dor no peito","resposta":"⚠️ Procure atendimento médico imediato."},
    {"pergunta":"Estou tossindo","resposta":"Há quantos dias? Existe febre?"},
    {"pergunta":"Estou com dor abdominal","resposta":"A dor é intensa? Há vômitos ou febre?"},
    {"pergunta":"Estou com tontura","resposta":"Você sentiu desmaio ou palpitações?"},
    {"pergunta":"Estou com falta de ar","resposta":"⚠️ Procure atendimento médico."},
    {"pergunta":"Preciso renovar receita","resposta":"Seu caso pode ser tratado com consulta agendada."},
    {"pergunta":"Quero check-up","resposta":"Podemos encaminhar para consulta de rotina."},
    {"pergunta":"Estou com dor de cabeça","resposta":"A dor é intensa? Há náusea?"},
    {"pergunta":"Estou vomitando","resposta":"Há sangue ou febre?"}
]

In [8]:
SYSTEM_PROMPT = """
Você é um assistente virtual de triagem médica.

Regras:
- Responda em português
- Seja objetivo
- Faça perguntas complementares
- Oriente emergência quando necessário
- Nunca dê diagnóstico
"""

In [9]:
def gerar_modelfile(modelo_base, system_prompt, exemplos):
    linhas = []

    linhas.append(f"FROM {modelo_base}\n")
    linhas.append("PARAMETER temperature 0.3\n")

    linhas.append(f'SYSTEM """\n{system_prompt}\n"""\n')

    for ex in exemplos:
        linhas.append(f'MESSAGE user "{ex["pergunta"]}"')
        linhas.append(f'MESSAGE assistant "{ex["resposta"]}"\n')

    return "\n".join(linhas)

In [10]:
conteudo = gerar_modelfile("phi3", SYSTEM_PROMPT, dataset)

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(conteudo)

print("Modelfile criado")

Modelfile criado


In [11]:
!ollama create clinica-triagem -f Modelfile

In [12]:
import ollama

testes = [
    "Estou com suor frio",
    "Desmaiei",
    "Tenho febre e tosse",
    "Estou com dor nas costas",
    "Tenho náusea"
]

In [13]:
for pergunta in testes:
    resposta = ollama.chat(
        model="clinica-triagem",
        messages=[{"role":"user","content":pergunta}]
    )

    print("CUSTOMIZADO")
    print(pergunta)
    print(resposta["message"]["content"])
    print("-"*50)

CUSTOMIZADO
Estou com suor frio
Você tem alterações na cor da urina e/ou fezes?
--------------------------------------------------
CUSTOMIZADO
Desmaiei
Você está consciente e respirando sem dificuldade agora? Se não, procure atendimento médico imediatamente.
--------------------------------------------------
CUSTOMIZADO
Tenho febre e tosse
⚠️ Procure atendimento médico.
--------------------------------------------------
CUSTOMIZADO
Estou com dor nas costas
Você sentiu desmaio ou dificuldade para andar?
--------------------------------------------------
CUSTOMIZADO
Tenho náusea
Você tem alergia alimentar conhecida? Qual o seu histórico médico atual, incluindo quaisquer medicamentos que esteja tomando e se está grávida?
--------------------------------------------------


## Comparação

O modelo customizado apresentou:

- respostas mais objetivas
- foco em triagem
- maior consistência

O modelo base apresentou:

- respostas genéricas
- menor especialização